In [1]:
import sys
import socket
import importlib
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from pathlib import Path

In [2]:
sys.path.append(str(Path.cwd().parent))

In [3]:
from src.config import raw_data_dir, processed_data_path, rdp_epsilon, target_features

In [4]:

data_path = raw_data_dir / "sample_data.npy"
sim = np.load(data_path, allow_pickle=True)

df = pd.DataFrame(sim)

In [5]:
# importlib.reload(sys.modules["src.preprocessing"])
from src.preprocessing import check_missing_values, drop_constant_columns, fit_preprocess_scalers, rdp



In [ ]:
df_rdp = rdp(df, epsilon=rdp_epsilon)
processed_data_path.parent.mkdir(parents=True, exist_ok=True)
np.save(processed_data_path, df_rdp.to_numpy())
print(f"Saved processed dataset to {processed_data_path}")

In [6]:
df.head()

,initial_mass,initial_z,star_age,mass,logR,logP,logRho,logT,luminosity,opacity,x_mass_fraction_H,y_mass_fraction_He,z_mass_fraction_metals,eps_nuc,eps_nuc_neu_total,eps_grav_nh,eps_grav,zone,q
0,16.5,0.02,1.124803e+07,15.672543,2.769037,2.632306,-8.726443,3.565941,56927.325018,0.001937,0.667269,0.319678,0.013054,0.042915,1.739978e-29,-0.003933,-2.621206,1.0,1.0
1,16.5,0.02,1.124803e+07,15.672543,2.769037,2.632306,-8.726443,3.565941,56927.325018,0.001937,0.667269,0.319678,0.013054,0.042915,1.739978e-29,-0.003933,-2.621206,2.0,1.0
2,16.5,0.02,1.124803e+07,15.672543,2.769037,2.632306,-8.726443,3.565941,56927.325018,0.001937,0.667269,0.319678,0.013054,0.042915,1.739978e-29,-0.003933,-2.621206,3.0,1.0
3,16.5,0.02,1.124803e+07,15.672543,2.769037,2.632306,-8.726443,3.565941,56927.325018,0.001937,0.667269,0.319678,0.013054,0.042915,1.739978e-29,-0.003933,-2.621205,4.0,1.0
4,16.5,0.02,1.124803e+07,15.672543,2.769037,2.632306,-8.726443,3.565941,56927.325018,0.001937,0.667269,0.319678,0.013054,0.042915,1.739978e-29,-0.003933,-2.621205,5.0,1.0


In [ ]:
check_missing_values(df_rdp)
print(f"Data shadata/processedpe after RDP vs original shape: {df_rdp.shape} vs {df.shape}")

df_reduced = drop_constant_columns(df_rdp)

df_subset = df_reduced[target_features]

df_scaled, scalers = fit_preprocess_scalers(
    df_subset,
    normalize=True,
    standardize=False,
)


In [14]:
# Isolate relevant features
df_visual = df[target_features + ['star_age']]

# Construct the 3D structure using groupby for computational efficiency
stratified_data = {
    age: group[target_features] 
    for age, group in df_visual.groupby('star_age')
}

# Extract the first available age to verify structure
first_age = list(stratified_data.keys())[0]

max_age_slice = max(stratified_data, key=lambda age: len(stratified_data[age]))
max_datapoints = len(stratified_data[max_age_slice])
min_age_slice = min(stratified_data, key=lambda age: len(stratified_data[age]))
min_datapoints = len(stratified_data[min_age_slice])

num_age_slices = len(stratified_data)


print(f"Time slice: {min_age_slice}")
print(f"Datapoint count: {min_datapoints}")
print(f"Time slice: {max_age_slice}")
print(f"Datapoint count: {max_datapoints}")
print(f"Total number of age slices: {num_age_slices}")

Time slice: 231089.89284351448
Datapoint count: 964
Time slice: 11098464.895993715
Datapoint count: 3350
Total number of age slices: 484


In [ ]:
def rdp(df, epsilon: float, rdp_feature):
    

def timestamps(df, rdp_feature):
	# create a list of dataframes at each time stamp
    stratified_data = {
        feature: group.drop(columns=[rdp_feature]) 
        for feature, group in df.groupby(rdp_feature)
    }

    # find slice with the minimum number of datapoints
    min_age_slice = min(stratified_data, key=lambda feature: len(stratified_data[feature]))
    min_datapoints = len(stratified_data[min_age_slice])

    return min_datapoints, stratified_data










In [ ]:
rdp_feature = 'time'
stratified_data = rdp(df_visual, .05, rdp_feature)

first_age = list(stratified_data.keys())[0]

max_age_slice = max(stratified_data, key=lambda feature: len(stratified_data[feature]))
max_datapoints = len(stratified_data[max_age_slice])
min_age_slice = min(stratified_data, key=lambda feature: len(stratified_data[feature]))
min_datapoints = len(stratified_data[min_age_slice])

num_age_slices = len(stratified_data)


print(f"Time slice: {min_age_slice}")
print(f"Datapoint count: {min_datapoints}")
print(f"Time slice: {max_age_slice}")
print(f"Datapoint count: {max_datapoints}")
print(f"Total number of age slices: {num_age_slices}")

KeyboardInterrupt: 

In [ ]:
# time = df_scaled['star_age'].to_numpy()[::10]
# mass = df_scaled['mass'].to_numpy()[::10]
# temp = df_scaled['logT'].to_numpy()[::10]

# fig = plt.figure(figsize=(10, 7))
# ax = fig.add_subplot(projection='3d')

# ax.plot_trisurf(time, mass, temp, cmap='viridis', edgecolor='none')
# ax.set_xlabel('Star Age')
# ax.set_ylabel('Mass')
# ax.set_zlabel('Log T')
# ax.set_title('Stellar Evolution Surface')
# plt.show()

In [ ]:
fig = go.Figure(
    data=[
        go.Mesh3d(
            x=time,
            y=mass,
            z=temp,
            intensity=temp,
            colorscale="Viridis",
            opacity=0.8,
        )
    ]
)
fig.update_layout(
    scene=dict(
        xaxis_title="Star Age",
        yaxis_title="Mass",
        zaxis_title="Log T",
    ),
    title="Stellar Evolution Surface",
)
fig.show()

In [ ]:
# importlib.reload(sys.modules["src.train"])
from src.train import train_model

In [ ]:
metrics = train_model()

In [ ]:
epochs = range(len(metrics["train_mse"]))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

ax1.plot(epochs, metrics["train_mse"], label="Train MSE")
ax1.plot(epochs, metrics["val_mse"], label="Validation MSE")
ax1.set_title("Reconstruction Loss (MSE)")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("MSE")
ax1.legend()
ax1.grid(True, linestyle="--", alpha=0.6)

ax2.plot(epochs, metrics["train_kld"], label="Train KLD")
ax2.plot(epochs, metrics["val_kld"], label="Validation KLD")
ax2.set_title("Latent Divergence (KLD)")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("KLD")
ax2.legend()
ax2.grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()